# Import libraries

In [ ]:
import sys
sys.path.append('/Users/BKieft/Metabolomics/metatlas2')
import metatlas2.database_interact as dbi
import metatlas2.lcmsruns_tools as lrt
import metatlas2.ms1_ms2_analysis as msa
import metatlas2.rt_align_tools as rat

In [ ]:
# Required: QC (model) and Target (reference) atlas for performing RT alignment
QC_ATLAS_UID = "atlas-c94e57c00d8245ebb72eef59c2862c8f"
TARGET_ATLAS_UID = "atlas-e1c4aea2f54f447dbb37531b6dabbe6f"
# Required: Project name
PROJECT_NAME = "20250707_JGI_MS_510172_SedRhizVegOut_final_IQX_HILICZ_USHXG02519"

In [ ]:
CONFIG = ldt.load_metatlas_config("/Users/BKieft/Metabolomics/metatlas2/configs/metatlas2_config.yaml")
PROJECT_FILES_PATH = f"{CONFIG['paths']['projects_dir']}/{PROJECT_NAME}/lcmsruns"
PROJECT_DB_PATH = f"{CONFIG['paths']['projects_dir']}/{PROJECT_NAME}/{PROJECT_NAME}.duckdb"

In [ ]:
dbi.create_project_database(PROJECT_DB_PATH, CONFIG)

In [ ]:
lcmsrun_files = dbi.save_lcmsruns_to_db(PROJECT_DB_PATH, PROJECT_NAME, PROJECT_FILES_PATH, CONFIG)

In [ ]:
matches_df, matching_stats = msa.extract_and_match_qc_compounds(PROJECT_DB_PATH, QC_ATLAS_UID, CONFIG)

In [ ]:
best_model, model_results, model_stats = rat.build_rt_alignment_model(matches_df, CONFIG)

In [ ]:
rt_summary, rt_atlas_uid = rat.apply_rt_correction_to_target(PROJECT_DB_PATH, TARGET_ATLAS_UID, best_model, lcmsrun_files, model_results)

In [ ]:
dbi.get_rt_correction_table_entry(PROJECT_DB_PATH, rt_atlas_uid)